In [4]:
import duckdb
import pandas as pd
import os
from datetime import datetime

# 설정
DB_PATH = '/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/duckdb/mimic_total.duckdb'

# DuckDB 연결
con = duckdb.connect(DB_PATH)

In [5]:
check_query = """SELECT
    itemid,
    label,
    category,
    unitname
FROM d_items
WHERE LOWER(label) LIKE '%urine%'
   OR LOWER(label) LIKE '%void%'
   OR LOWER(label) LIKE '%foley%'
;
"""

df_check = con.execute(check_query).df()

In [3]:
df_check

,itemid,label,category,unitname
0,220473,Taurine,Ingredients - general (Not In Use),mg
1,220799,ZSpecific Gravity (urine),Labs,None
2,224015,Urine Source,GI/GU,None
3,224016,Urine Color,GI/GU,None
4,224876,Urine Appearance,GI/GU,None
5,225454,Urine Culture,6-Cultures,None
6,226559,Foley,Output,mL
7,226560,Void,Output,mL
8,226566,Urine and GU Irrigant Out,Output,mL
9,226627,OR Urine,Output,mL


In [10]:
check_query2= """SELECT
    itemid,
    label,
    category,
    unitname
FROM d_items
WHERE itemid In(220045,
220210,
220277,
223761,
223762,
220179,
220180,
220181,
220050,
220051,
220052,
226559,
226560)
"""

df_check2 = con.execute(check_query2).df()

In [11]:
df_check2

,itemid,label,category,unitname
0,220045,Heart Rate,Routine Vital Signs,bpm
1,220050,Arterial Blood Pressure systolic,Routine Vital Signs,mmHg
2,220051,Arterial Blood Pressure diastolic,Routine Vital Signs,mmHg
3,220052,Arterial Blood Pressure mean,Routine Vital Signs,mmHg
4,220179,Non Invasive Blood Pressure systolic,Routine Vital Signs,mmHg
5,220180,Non Invasive Blood Pressure diastolic,Routine Vital Signs,mmHg
6,220181,Non Invasive Blood Pressure mean,Routine Vital Signs,mmHg
7,220210,Respiratory Rate,Respiratory,insp/min
8,220277,O2 saturation pulseoxymetry,Respiratory,%
9,223761,Temperature Fahrenheit,Routine Vital Signs,°F


In [23]:
check_query3 = """
SELECT DISTINCT category 
FROM d_items 
WHERE LOWER(label) LIKE '%norepinephrine%'
   OR LOWER(label) LIKE '%dopamine%'
   OR LOWER(label) LIKE '%vasopressin%'
   OR LOWER(label) LIKE '%ephedrine%'
;
"""
df_categories = con.execute(check_query3).df()
print(df_categories)

      category
0  Medications


In [36]:
# 2. 특정 승압제들의 itemid 찾기
pressor_search = """
SELECT itemid, label, category, linksto, unitname
FROM d_items
WHERE linksto = 'inputevents'
  AND (
    LOWER(label) LIKE '%norepinephrine%'
    OR LOWER(label) LIKE '%epinephrine%'
    OR LOWER(label) LIKE '%vasopressin%'
    OR LOWER(label) LIKE '%dopamine%'
  )
;
"""
df_pressor_items = con.execute(pressor_search).df()
print(df_pressor_items)

   itemid           label     category      linksto unitname
0  221289     Epinephrine  Medications  inputevents       mg
1  221662        Dopamine  Medications  inputevents       mg
2  221906  Norepinephrine  Medications  inputevents       mg
3  222315     Vasopressin  Medications  inputevents    units
4  229617    Epinephrine.  Medications  inputevents       mg


In [40]:
# Epinephrine vs Epinephrine. 비교
epi_comparison_query = """
SELECT *
FROM d_items
WHERE itemid IN (221289, 229617)
ORDER BY itemid
;
"""

df_epi_comparison = con.execute(epi_comparison_query).df()
print(df_epi_comparison)

   itemid         label  abbreviation      linksto     category unitname  \
0  221289   Epinephrine   Epinephrine  inputevents  Medications       mg   
1  229617  Epinephrine.  Epinephrine.  inputevents  Medications       mg   

  param_type lownormalvalue highnormalvalue  
0   Solution           None            None  
1   Solution           None            None  


In [43]:
# 간단 비교
simple_query = """
SELECT 
    itemid,
    COUNT(*) AS count,
    COUNT(DISTINCT subject_id) AS patients
FROM inputevents
WHERE itemid IN (221289, 229617)
GROUP BY itemid
;
"""

df_simple = con.execute(simple_query).df()
print(df_simple)

if len(df_simple) == 2:
    if df_simple.iloc[0]['count'] == df_simple.iloc[1]['count']:
        print("\n✅ 두 itemid의 사용 횟수가 동일합니다")
    else:
        print("\n❌ 두 itemid의 사용 횟수가 다릅니다")
elif len(df_simple) == 1:
    print(f"\n⚠️ itemid {df_simple.iloc[0]['itemid']}만 사용 기록이 있습니다")
else:
    print("\n❌ 두 itemid 모두 사용 기록이 없습니다")

   itemid  count  patients
0  229617    234       106
1  221289  31495      3706

❌ 두 itemid의 사용 횟수가 다릅니다


In [46]:
# rate, amount 분포 비교 (타입 캐스팅 추가)
dose_query = """
SELECT 
    itemid,
    AVG(CAST(amount AS DOUBLE)) AS avg_amount,
    MIN(CAST(amount AS DOUBLE)) AS min_amount,
    MAX(CAST(amount AS DOUBLE)) AS max_amount,
    AVG(CAST(rate AS DOUBLE)) AS avg_rate,
    AVG(CAST(originalrate AS DOUBLE)) AS avg_original_rate
FROM inputevents
WHERE itemid IN (221289, 229617)
  AND amount IS NOT NULL
  AND rate IS NOT NULL
GROUP BY itemid
;
"""
df_dose = con.execute(dose_query).df()
print(df_dose)

   itemid  avg_amount  min_amount   max_amount  avg_rate  avg_original_rate
0  221289     1.07325    0.000112  1707.178021  0.154263           0.153513


In [47]:
# 229617의 실제 데이터 샘플 확인
sample_query = """
SELECT 
    itemid,
    amount,
    amountuom,
    rate,
    rateuom,
    originalrate,
    starttime
FROM inputevents
WHERE itemid = 229617
LIMIT 10
;
"""
df_sample = con.execute(sample_query).df()
print("=== Epinephrine. (229617) 샘플 데이터 ===")
print(df_sample)

# NULL 값 비율 확인
null_query = """
SELECT 
    itemid,
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    COUNT(rate) AS has_rate,
    COUNT(originalrate) AS has_originalrate,
    ROUND(100.0 * COUNT(amount) / COUNT(*), 2) AS amount_fill_rate
FROM inputevents
WHERE itemid IN (221289, 229617)
GROUP BY itemid
;
"""
df_null = con.execute(null_query).df()
print("\n=== NULL 값 비율 ===")
print(df_null)

=== Epinephrine. (229617) 샘플 데이터 ===
   itemid amount amountuom  rate rateuom originalrate            starttime
0  229617      1        mg  None    None            1  2167-03-31 07:14:00
1  229617      1        mg  None    None            1  2190-04-21 09:00:00
2  229617      1        mg  None    None            1  2195-07-23 19:25:00
3  229617      1        mg  None    None            1  2195-07-24 23:03:00
4  229617    0.1        mg  None    None            0  2163-09-21 21:13:00
5  229617    0.5        mg  None    None          0.5  2133-08-27 17:23:00
6  229617    0.5        mg  None    None          0.5  2138-08-11 17:09:00
7  229617    0.5        mg  None    None          0.5  2138-08-11 17:11:00
8  229617      1        mg  None    None            1  2150-02-14 13:15:00
9  229617      1        mg  None    None            1  2155-11-09 22:30:00

=== NULL 값 비율 ===
   itemid  total  has_amount  has_rate  has_originalrate  amount_fill_rate
0  229617    234         234         0      

핵심 차이점:
항목Epinephrine (221289)Epinephrine. (229617)
rate 값✅ 있음 (100% 채워짐)❌ 모두 None투여 방식지속 주입 (infusion)일회성 볼루스 (bolus)사용 빈도31,495회234회
결론:

221289 (Epinephrine): rate가 있어서 지속적으로 천천히 주입하는 방식 (예: 시간당 0.15mg)
229617 (Epinephrine.): rate가 None이고 amount만 있어서 일회성으로 빠르게 투여하는 볼루스 (예: 1mg 단발성 주사)

In [49]:
# Systolic BP만 비교 (NBP vs ABP)
simple_bp_query = """
SELECT 
    CASE 
        WHEN itemid = 220179 THEN 'NBP Systolic'
        WHEN itemid = 220050 THEN 'ABP Systolic'
    END as bp_type,
    COUNT(*) as count,
    COUNT(DISTINCT stay_id) as patients,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as percentage
FROM chartevents
WHERE itemid IN (220179, 220050)
GROUP BY itemid
;
"""

df_simple_bp = con.execute(simple_bp_query).df()
print(df_simple_bp)


        bp_type    count  patients  percentage
0  ABP Systolic  3087686     36028       36.47
1  NBP Systolic  5378740     93145       63.53
